# Hyperband Search for Tiny Shakespeare

Goal: test whether Hyperband-style search beats manual autoresearch sweep under same fixed compute budget.

In [3]:
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

REQUIRED_MODULES = ["numpy", "torch", "matplotlib"]
missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing) + "\n"
        + "Install first, e.g. pip install numpy torch matplotlib notebook"
    )

import torch

ROOT = Path("/content/comp560-niloy/autoresearch-tinyshakespeare")
REPO = ROOT.parent

if not ROOT.exists():
    print("Cloning repo into /content ...")
    subprocess.run(
        ["git", "clone", "https://github.com/niloy-saha-123/comp560-niloy.git", str(REPO)],
        check=True,
    )

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd:", Path.cwd().resolve())
print("root:", ROOT)
print(sys.executable)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

for required in ["prepare.py", "train.py", "hyperband_utils.py", "compare_seeds.py", "compare_budgets.py"]:
    print(required, (ROOT / required).exists())

generated = ["input.txt", "meta.json", "eval_batches.npz"]
missing_generated = [name for name in generated if not (ROOT / name).exists()]
print("generated status:", {name: (ROOT / name).exists() for name in generated})

if missing_generated:
    print("Running prepare.py because missing:", missing_generated)
    subprocess.run([sys.executable, str(ROOT / "prepare.py")], cwd=ROOT, check=True)
    print("prepare.py done")


cwd: /content/comp560-niloy/autoresearch-tinyshakespeare
root: /content/comp560-niloy/autoresearch-tinyshakespeare
/usr/bin/python3
torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA A100-SXM4-80GB
prepare.py True
train.py True
hyperband_utils.py False
compare_seeds.py True
compare_budgets.py True
generated status: {'input.txt': False, 'meta.json': False, 'eval_batches.npz': False}
Running prepare.py because missing: ['input.txt', 'meta.json', 'eval_batches.npz']
prepare.py done


In [ ]:
from pprint import pprint

from hyperband_utils import SEARCH_SPACE, HEAD_MAP, bracket_schedule

print("search space")
pprint(SEARCH_SPACE)
print("head map")
pprint(HEAD_MAP)

for s in [2, 1, 0]:
    print(f"bracket s={s}")
    pprint(bracket_schedule(s=s, eta=3, r=30, R=270))

In [ ]:
from hyperband_utils import (
    BASELINE_CONFIG,
    MANUAL_BEST_CONFIG,
    bracket_winner_table,
    compare_reference_configs,
    compute_best_so_far,
    plot_best_so_far,
    plot_budget_comparison,
    read_jsonl,
    run_successive_halving_bracket,
    select_hyperband_best_config,
    summarize_by_budget,
    write_bracket_log,
    write_hyperband_summary,
)

ARTIFACTS = ROOT / "hyperband_artifacts"
BRACKET_LOG_PATH = ARTIFACTS / "hyperband_brackets.json"
HYPERBAND_SUMMARY_PATH = ROOT / "HYPERBAND_SUMMARY.md"

TRAIN_SEED = 1337
BRACKET_SEEDS = {2: 2002, 1: 2001, 0: 2000}
COMPARE_BUDGETS = [60, 120, 240]
COMPARE_SEEDS = [1337, 2024, 7]

print("artifacts:", ARTIFACTS)

In [ ]:
rows_s2, log_s2 = run_successive_halving_bracket(
    s=2,
    artifacts_root=ARTIFACTS,
    bracket_seed=BRACKET_SEEDS[2],
    train_seed=TRAIN_SEED,
    eta=3,
    r=30,
    R=270,
    verbose=True,
)
print("s=2 runs:", len(rows_s2))

In [ ]:
rows_s1, log_s1 = run_successive_halving_bracket(
    s=1,
    artifacts_root=ARTIFACTS,
    bracket_seed=BRACKET_SEEDS[1],
    train_seed=TRAIN_SEED,
    eta=3,
    r=30,
    R=270,
    verbose=True,
)
print("s=1 runs:", len(rows_s1))

In [ ]:
rows_s0, log_s0 = run_successive_halving_bracket(
    s=0,
    artifacts_root=ARTIFACTS,
    bracket_seed=BRACKET_SEEDS[0],
    train_seed=TRAIN_SEED,
    eta=3,
    r=30,
    R=270,
    verbose=True,
)
print("s=0 runs:", len(rows_s0))

In [ ]:
all_rows = rows_s2 + rows_s1 + rows_s0
bracket_logs = [log_s2, log_s1, log_s0]
write_bracket_log(bracket_logs, BRACKET_LOG_PATH)

best_so_far = compute_best_so_far(all_rows)
winners = bracket_winner_table(all_rows)
hyperband_best_config = select_hyperband_best_config(all_rows, require_max_budget=True)

print("total rows:", len(all_rows))
print("best-so-far tail")
for row in best_so_far[-5:]:
    print(row)

print("bracket winners")
for row in winners:
    print(row)

print("selected Hyperband config")
print(json.dumps(hyperband_best_config, indent=2))

In [ ]:
compare_rows = compare_reference_configs(
    config_map={
        "baseline": BASELINE_CONFIG,
        "manual_best": MANUAL_BEST_CONFIG,
        "hyperband_best": hyperband_best_config,
    },
    budgets=COMPARE_BUDGETS,
    seeds=COMPARE_SEEDS,
    artifacts_root=ARTIFACTS,
    verbose=True,
)

summary_rows = summarize_by_budget(compare_rows, ["baseline", "manual_best", "hyperband_best"])
for row in summary_rows:
    print(row)

In [ ]:
plot_best_so_far(all_rows)
plot_budget_comparison(summary_rows)

write_hyperband_summary(
    search_rows=all_rows,
    compare_rows=compare_rows,
    hyperband_best_config=hyperband_best_config,
    output_path=HYPERBAND_SUMMARY_PATH,
)

print("wrote:", HYPERBAND_SUMMARY_PATH)
print(HYPERBAND_SUMMARY_PATH.read_text(encoding="utf-8"))